# Dispersion as a Statistic: Juilland's D and Gries's DP

**Computational Text Analysis Pack · Module 10 — Visualising Text**

*Technique:* Computing dispersion; why DP replaced D; the frequency confound

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/10d_dispersion_statistics.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Notebook 10c drew dispersion plots — where in a text a word appears. A plot is
a good way to see dispersion for one word, and useless for ranking a thousand.
For that you need a statistic, and the field has argued about which one.

**Juilland's D** was the standard for decades. **Gries's DP** (Deviation of
Proportions, 2008) has largely replaced it, because Biber et al. (2016) and
Gries (2020, 2022) showed D loses discriminating range once a corpus is split
into many parts — which is what any modern corpus is.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
work = (df_colonial.groupby("work_id")["text"].apply(" ".join)
                    .sort_values(key=lambda s: s.str.len(), ascending=False))
toks = tokenise(work.iloc[0])
print(f"Working on one long work: {len(toks):,} tokens")

def parts_of(tokens, n_parts):
    size = len(tokens) // n_parts
    return [tokens[i * size:(i + 1) * size] for i in range(n_parts)]

## The two statistics

Both compare a word's *observed* distribution across corpus parts against the
distribution you would expect if it were spread in proportion to part size.

- **Juilland's D** = `1 - (sd / mean) / sqrt(n-1)`, on the per-part rates.
- **Gries's DP** = half the total absolute difference between observed and
  expected proportions. It runs 0 (perfectly even) to nearly 1 (all in one part).

Note they run in opposite directions: high D means even, high DP means clumped.

In [ ]:
def juilland_d(freqs, sizes):
    rates = np.array([f / s for f, s in zip(freqs, sizes)])
    if rates.mean() == 0:
        return np.nan
    cv = rates.std(ddof=1) / rates.mean()
    return 1 - cv / np.sqrt(len(rates) - 1)

def gries_dp(freqs, sizes):
    freqs, sizes = np.array(freqs, float), np.array(sizes, float)
    if freqs.sum() == 0:
        return np.nan
    observed = freqs / freqs.sum()
    expected = sizes / sizes.sum()
    return float(np.abs(observed - expected).sum() / 2)

chunks = parts_of(toks, 20)
sizes = [len(c) for c in chunks]
per_chunk = [Counter(c) for c in chunks]
total = Counter(toks)

rows = []
for w, n in total.most_common(400):
    freqs = [c[w] for c in per_chunk]
    rows.append({"word": w, "freq": n,
                 "Juilland_D": juilland_d(freqs, sizes),
                 "Gries_DP": gries_dp(freqs, sizes)})
disp = pd.DataFrame(rows)
print("Most evenly spread (low DP) — the function-word core:")
print(disp.nsmallest(8, "Gries_DP").to_string(index=False,
                                              float_format=lambda v: f"{v:.3f}"))
print()
print("Most clumped (high DP) — words tied to one episode or chapter:")
print(disp.nlargest(8, "Gries_DP").to_string(index=False,
                                             float_format=lambda v: f"{v:.3f}"))

## The complaint against Juilland's D

Gries's objection is about *range*: split a corpus into more parts and D bunches
up near the top, so it stops separating words that are genuinely differently
distributed. DP keeps its spread. That is testable directly.

In [ ]:
print(f"{'parts':>7}{'D range':>12}{'D sd':>10}{'DP range':>12}{'DP sd':>10}")
print("-" * 51)
for n_parts in [5, 10, 20, 50, 100]:
    ch = parts_of(toks, n_parts)
    sz = [len(c) for c in ch]
    cs = [Counter(c) for c in ch]
    ds, dps = [], []
    for w, _ in total.most_common(300):
        f = [c[w] for c in cs]
        ds.append(juilland_d(f, sz))
        dps.append(gries_dp(f, sz))
    ds = np.array([d for d in ds if not np.isnan(d)])
    dps = np.array([d for d in dps if not np.isnan(d)])
    print(f"{n_parts:>7}{ds.max() - ds.min():>12.3f}{ds.std():>10.3f}"
          f"{dps.max() - dps.min():>12.3f}{dps.std():>10.3f}")
print()
print("As the corpus is cut into more parts, D's spread collapses while DP's")
print("holds. With 100 parts D can no longer tell the words apart.")

## The frequency confound

Gries (2022) then complicated his own measure: most dispersion statistics are
partly confounded with frequency, so a rare word looks clumped simply because it
is rare. Before using DP as evidence, check how much of it is frequency.

In [ ]:
r = disp["Gries_DP"].corr(np.log(disp["freq"]))
print(f"correlation of DP with log frequency: {r:.3f}")
print()
plt.figure(figsize=(6.5, 4))
plt.scatter(disp["freq"], disp["Gries_DP"], s=10, alpha=.5)
plt.xscale("log")
plt.xlabel("frequency (log)")
plt.ylabel("Gries's DP")
plt.title("Rare words look clumped because they are rare")
plt.tight_layout()
print("A dispersion claim about a rare word needs a frequency-matched")
print("comparison, or it is a frequency claim wearing a dispersion label.")

### 🔧 Your turn 1

Compute DP across *languages* in the news corpus instead of across parts of one book. Which words are evenly spread across all 16?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Words even across all sixteen languages are mostly proper nouns, numerals and
borrowings — *Nigeria*, *COVID*, *Facebook* — because everything else is
language-specific by definition. This is dispersion used as a *comparative*
instrument rather than a within-text one, and it is a good way to find the
shared referential layer of a multilingual corpus. Note the parts here differ
enormously in size, which is precisely the case DP handles and raw variance does
not.

</details>